# 06 — First spatial geometry features (P2 prototype)

**Purpose:** Convert the 20-match canonical decision records into strictly contemporaneous, interpretable spatial measurements. This is the **first spatial feature baseline**, *not* a frozen operational pressure metric or a finished test of H1/H3.

## Scientific rules
- The player-possession start frame is the observation time. Distance calculations use only coordinate information stored at that frame.
- We measure proximity, **not inferred defensive intention**, pressing effectiveness, defensive cohesion, passing feasibility or actual pressure. These require more information.
- Do not use outcome labels (`carry_realized`, `end_type`) or phase labels as predictive inputs here.
- Retain two parallel analyses: (1) valid detected **and** extrapolated coordinates; (2) strictly detected participants. Missing detections are not zero distance.
- The nominal 3 m and 5 m proximity bands are **illustrative exploratory cutoffs** to support scientific inspection, not validated tactical thresholds. Do not select a 'best' radius by looking at the outcomes.
- One `match_id` is a cross-validation group. Do not treat rows/frames as statistically independent.

**Expected input:** `outputs/P1_canonical_decisions_match_{match_id}_ROLES.jsonl` generated by Notebook 05. All outputs stay inside `outputs/`.


In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import csv, json, math, statistics

ROOT=next((p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents) if (p/'data'/'matches').is_dir()),None)
if ROOT is None: raise FileNotFoundError("Run inside the SkillCorner repository")
OUT=ROOT/'outputs'; OUT.mkdir(exist_ok=True)
FILES=sorted(OUT.glob('P1_canonical_decisions_match_*_ROLES.jsonl'))
if len(FILES)!=20:
    raise AssertionError(f'Expected 20 Notebook 05 output files, found {len(FILES)}. Rerun Notebook 05 if required.')
print('Input files:',len(FILES))


Input files: 20


## 1. Observable proximity metrics

- `nearest_defender_m`: nearest opponent to the carrier, with valid coordinates (detected or extrapolated).
- `defenders_within_3m` and `defenders_within_5m`: simple local proximity counts.
- `nearest_teammate_m`: nearest supporting teammate, irrespective of whether a pass is actually feasible.
- `nearest_defender_detected_only_m`: a sensitivity indicator that uses only directly detected positions for both the carrier and defenders.

Tracking coordinates must be in metres. The previous pitch diagnostics suggest metre-scale coordinates; this notebook performs a **bounded sanity check** on computed separations. It does not establish orientation or pitch-side semantics.


In [2]:
def xy_valid(xy):
    return isinstance(xy,list) and len(xy)==2 and all(isinstance(v,(int,float)) and math.isfinite(v) for v in xy)
def dist(a,b):return math.hypot(a[0]-b[0],a[1]-b[1])
def positions(row,role,quality):
    return [p['xy'] for p in row.get(f'{role}_{quality}',[]) if xy_valid(p.get('xy'))]
def one_feature_row(r):
    base={k:r.get(k) for k in ('match_id','period','frame','possession_event_id','carrier_player_id','attacking_team_id','phase_status')}
    carrier=r.get('carrier_xy')
    base['carrier_xy_valid']=xy_valid(carrier)
    base['carrier_detected']=r.get('carrier_detected') is True
    base['ball_xy_valid']=xy_valid(r.get('ball_xy'))
    base['ball_detected']=r.get('ball_detected') is True
    for role in ('teammates','opponents'):
        det=positions(r,role,'detected')
        ext=positions(r,role,'extrapolated')
        base[f'{role}_detected_count']=len(det)
        base[f'{role}_extrapolated_count']=len(ext)
        base[f'{role}_position_count']=len(det)+len(ext)
    base['nearest_defender_m']=None
    base['nearest_teammate_m']=None
    base['defenders_within_3m']=None
    base['defenders_within_5m']=None
    base['nearest_defender_detected_only_m']=None
    base['defenders_detected_within_3m']=None
    if not base['carrier_xy_valid']:
        return base
    opp_det=positions(r,'opponents','detected')
    opp=opp_det+positions(r,'opponents','extrapolated')
    mate=positions(r,'teammates','detected')+positions(r,'teammates','extrapolated')
    ds=[dist(carrier,p) for p in opp]
    ts=[dist(carrier,p) for p in mate]
    if ds:
        base['nearest_defender_m']=min(ds)
        base['defenders_within_3m']=sum(d<=3.0 for d in ds)
        base['defenders_within_5m']=sum(d<=5.0 for d in ds)
    if ts:base['nearest_teammate_m']=min(ts)
    if base['carrier_detected'] and opp_det:
        dd=[dist(carrier,p) for p in opp_det]
        base['nearest_defender_detected_only_m']=min(dd)
        base['defenders_detected_within_3m']=sum(d<=3.0 for d in dd)
    return base


## 2. Process and save one geometry file for all 20 matches

This stage reads **only the already-created decision snapshots**, not the full tracking streams again. It writes *feature-only* JSONL to avoid accidentally mixing retrospective outcomes with observation-time geometry.


In [3]:
stats=[]
all_nearest=[]
for file in FILES:
    mid=file.name.split('_match_')[1].split('_ROLES')[0]
    output=OUT/f'P2_spatial_features_match_{mid}.jsonl'
    counters=Counter()
    examples=[]
    with file.open(encoding='utf-8') as stream,output.open('w',encoding='utf-8') as target:
        for line in stream:
            r=json.loads(line)
            f=one_feature_row(r)
            counters['rows']+=1
            for key in ('carrier_xy_valid','carrier_detected','ball_xy_valid'):
                counters[key]+=int(f[key])
            counters['nearest_defender_valid']+=int(f['nearest_defender_m'] is not None)
            counters['nearest_defender_detected_only_valid']+=int(f['nearest_defender_detected_only_m'] is not None)
            counters['nearest_teammate_valid']+=int(f['nearest_teammate_m'] is not None)
            counters['rows_without_any_opponent_position']+=int(f['opponents_position_count']==0)
            if f['nearest_defender_m'] is not None:
                all_nearest.append(f['nearest_defender_m'])
                if f['nearest_defender_m']>70 and len(examples)<5:
                    examples.append({'period':f['period'],'frame':f['frame'],'nearest_defender_m':f['nearest_defender_m']})
            target.write(json.dumps(f,separators=(',',':'))+'\n')
    stats.append({'match_id':mid,**dict(counters),'large_distance_examples':examples})
    print(mid,'rows',counters['rows'],'nearest defender valid',counters['nearest_defender_valid'],
          'direct-detection-only',counters['nearest_defender_detected_only_valid'])


1874553 rows 959 nearest defender valid 956 direct-detection-only 934
1886347 rows 999 nearest defender valid 999 direct-detection-only 967
1899585 rows 929 nearest defender valid 929 direct-detection-only 896
1925299 rows 1062 nearest defender valid 1062 direct-detection-only 1003
1927964 rows 1140 nearest defender valid 1139 direct-detection-only 1102
1953632 rows 979 nearest defender valid 970 direct-detection-only 939
1959846 rows 946 nearest defender valid 945 direct-detection-only 920
1986691 rows 915 nearest defender valid 915 direct-detection-only 886
1996435 rows 1005 nearest defender valid 997 direct-detection-only 973
1996436 rows 786 nearest defender valid 785 direct-detection-only 771
2006229 rows 979 nearest defender valid 977 direct-detection-only 951
2006363 rows 976 nearest defender valid 976 direct-detection-only 942
2007448 rows 863 nearest defender valid 862 direct-detection-only 831
2007721 rows 959 nearest defender valid 959 direct-detection-only 929
2010085 rows 

## 3. Eligibility report and stopping gate

A geometry observation is eligible for a carrier-to-defender distance only when both have finite coordinates and at least one opponent coordinate exists. Whether the position is directly observed is stored separately. The two quality regimes provide a future **sensitivity comparison**, not independent samples.

A very large distance value may be a valid long-range separation, but values above 70 m are surfaced as an inspection trigger. No rows are silently removed or altered.


In [4]:
totals={key:sum(r.get(key,0) for r in stats) for key in (
 'rows','carrier_xy_valid','carrier_detected','ball_xy_valid','nearest_defender_valid',
 'nearest_defender_detected_only_valid','nearest_teammate_valid','rows_without_any_opponent_position')}
report={
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'purpose':'First pre-action spatial proximity features — exploratory',
 'source':'Notebook 05 canonical decision snapshots; match-grouped rows',
 'matches_processed':len(stats),
 'totals':totals,
 'nearest_defender_m_descriptive':({'min':min(all_nearest),'median':statistics.median(all_nearest),
       'max':max(all_nearest)} if all_nearest else None),
 'methodological_constraints':[
   'Player-possession start frame only; no future frames or observed outcomes used',
   'Proximity is not pressing, causation, path feasibility, pitch control or coordination',
   '3m and 5m are exploratory pre-specified inspection bands, not optimized thresholds',
   'Mixed detected/extrapolated positions compared with detected-only sensitivity subset',
   'No attacking-direction normalization and no hypothetical carry opportunity computation',
   'No predictive model or statistical inference trained'],
 'match_results':stats
}
p=OUT/'P2_spatial_geometry_gate.json'
p.write_text(json.dumps(report,indent=2),encoding='utf-8')
assert totals['rows']==18792, f"Unexpected row count {totals['rows']}; confirm same 20 match files"
print(json.dumps({'matches':len(stats),'totals':totals,'nearest_defender_descriptive':report['nearest_defender_m_descriptive']},indent=2))
print('Upload:',p)


{
  "matches": 20,
  "totals": {
    "rows": 18792,
    "carrier_xy_valid": 18757,
    "carrier_detected": 18092,
    "ball_xy_valid": 18758,
    "nearest_defender_valid": 18757,
    "nearest_defender_detected_only_valid": 18080,
    "nearest_teammate_valid": 18757,
    "rows_without_any_opponent_position": 34
  },
  "nearest_defender_descriptive": {
    "min": 0.05830951894845024,
    "median": 5.63490017657811,
    "max": 45.815499560738175
  }
}
Upload: /Users/fadihaddad/Documents/GitHub/SkillCorner_opendata_soccer/outputs/P2_spatial_geometry_gate.json
